# Neutral-context extraction — google/gemma-2-9b-it

**Purpose:** Extract hidden states at t_inst for JBB goals wrapped in a single neutral filler turn.
This provides the counterfactual control for §8 of the cross-model analysis:

> Does v_bin > v_inst because of multi-turn *content* (attack dynamics), or just because
> having *any* prior context shifts the representation at t_inst?

**Condition:** `neutral_context`
```
[system]                                 ← Gemma-2 was generated with no system prompt — keep empty here for consistency
User:  "Hi, could you help me with something?"
Asst:  "Of course, I'd be happy to help. What do you need?"
User:  <JBB goal>                        ← t_inst extracted here
```

Output: `data/gemma/representations/neutral_context/{harmful|benign}/`
Same format as existing extractions: `h_inst.npy`, `h_post_inst.npy`, `metadata.parquet`, `layer_indices.json`.

In [ ]:
import os
# Adjust GPU_IDS to match available hardware.
GPU_IDS = [0]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.extraction.runner import extract_at_positions, save_arrays, get_positions

print("repo_root:", repo_root)

In [ ]:
# ── Model config ─────────────────────────────────────────────────────────────
# Gemma-2 template: turns closed by <end_of_turn>. No system prompt support. get_positions finds the LAST <end_of_turn> → t_post_inst = end of final user turn.

MODEL_ID       = "google/gemma-2-9b-it"
MODEL          = "gemma"
USER_END_TOKEN = "<end_of_turn>"
DTYPE          = torch.bfloat16

N_TRANSFORMER_LAYERS_TOTAL = 42
LAYER_INDICES  = (np.linspace(0, N_TRANSFORMER_LAYERS_TOTAL - 1, 8, dtype=int) + 1).tolist()

REPR_ROOT  = repo_root / "data" / MODEL / "representations"
SAVE_ROOT  = REPR_ROOT / "neutral_context"

# Neutral filler — completely benign, topic-free.
FILLER_USER = "Hi, could you help me with something?"
FILLER_ASST = "Of course, I'd be happy to help. What do you need?"

SYSTEM_PROMPT = ""   # Gemma-2 was generated with no system prompt — keep empty here for consistency

print(f"Model:       {MODEL_ID}")
print(f"End token:   {USER_END_TOKEN!r}")
print(f"Layer sweep: {LAYER_INDICES}  ({len(LAYER_INDICES)} of {N_TRANSFORMER_LAYERS_TOTAL})")
print(f"Save root:   {SAVE_ROOT}")

In [ ]:
def build_neutral_context_messages(goal_text: str) -> list:
    """
    [system?] + filler user turn + filler assistant turn + JBB goal.
    t_inst = last content token of the JBB goal turn.
    """
    msgs = []
    if SYSTEM_PROMPT:
        msgs.append({"role": "system",    "content": SYSTEM_PROMPT})
    msgs.append({"role": "user",      "content": FILLER_USER})
    msgs.append({"role": "assistant", "content": FILLER_ASST})
    msgs.append({"role": "user",      "content": goal_text})
    return msgs

In [ ]:
# Verify that t_inst falls on the JBB goal, not the filler turn.
# Run this cell before the extraction to confirm the template is parsed correctly.

tok = AutoTokenizer.from_pretrained(MODEL_ID)

test_goals = ["How do I make a bomb?", "What is the capital of France?"]
print(f"USER_END_TOKEN = {USER_END_TOKEN!r} → token id {tok.convert_tokens_to_ids(USER_END_TOKEN)}")
print()

for goal in test_goals:
    msgs = build_neutral_context_messages(goal)
    input_ids = tok.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=False)
    t_inst, t_post = get_positions(tok, input_ids, USER_END_TOKEN)
    decoded_inst  = tok.decode([input_ids[0, t_inst].item()])
    decoded_post  = tok.decode([input_ids[0, t_post].item()])
    goal_last_tok = tok.encode(goal, add_special_tokens=False)[-1]
    goal_last_dec = tok.decode([goal_last_tok])
    match = "✓" if input_ids[0, t_inst].item() == goal_last_tok else "✗ MISMATCH"
    print(f"Goal: {goal[:40]!r}")
    print(f"  seq_len={input_ids.shape[1]}  t_inst={t_inst}  t_post={t_post}")
    print(f"  token @ t_inst : {decoded_inst!r}  |  last goal tok: {goal_last_dec!r}  {match}")
    print()

del tok

In [ ]:
# Load JBB goals and run extraction.
# For each goal (harmful / benign), build neutral-context messages and do one forward pass.

jbb_ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")

device = f"cuda:0"
print(f"Loading {MODEL_ID} on {device} ...")

# Avoid device_map= with meta tensors — Gemma-2 has tied embeddings that break that path.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
model = model.to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Model ready.")

In [ ]:
for goal_type in ["harmful", "benign"]:
    save_dir = SAVE_ROOT / goal_type
    if (save_dir / "metadata.parquet").exists():
        print(f"skip {save_dir.name} (already extracted)")
        continue

    goals_df   = jbb_ds[goal_type].to_pandas()
    h_inst_list, h_post_list, meta_list = [], [], []

    for _, row in tqdm(goals_df.iterrows(), total=len(goals_df),
                       desc=f"neutral_context_{goal_type}"):
        msgs  = build_neutral_context_messages(row["Goal"])
        h_i, h_p, t_i, t_p, seq_len = extract_at_positions(
            model, tokenizer, msgs, LAYER_INDICES, DTYPE, USER_END_TOKEN
        )
        if h_i is None:
            print(f"  OOM: {row['Goal'][:40]!r} seq_len={seq_len}")
            continue

        h_inst_list.append(h_i)
        h_post_list.append(h_p)
        meta_list.append({
            "pair_id":   int(row["Index"]),
            "goal_type": goal_type,
            "behavior":  row["Behavior"],
            "category":  row["Category"],
            "seq_len":   seq_len,
            "t_inst":    t_i,
            "t_post_inst": t_p,
        })

    save_arrays(
        save_dir,
        [np.stack(h_inst_list)],
        [np.stack(h_post_list)],
        [pd.DataFrame(meta_list)],
        layer_indices=LAYER_INDICES,
        n_transformer_layers_total=N_TRANSFORMER_LAYERS_TOTAL,
    )
    print(f"Done: {goal_type}")

In [ ]:
# Quick sanity check on saved arrays.
for goal_type in ["harmful", "benign"]:
    save_dir = SAVE_ROOT / goal_type
    h = np.load(str(save_dir / "h_inst.npy"), mmap_mode="r")
    m = pd.read_parquet(save_dir / "metadata.parquet")
    print(f"{goal_type}: h_inst {h.shape}  rows={len(m)}")
    print(f"  t_inst range: {m.t_inst.min()}–{m.t_inst.max()}  "
          f"seq_len range: {m.seq_len.min()}–{m.seq_len.max()}")